In [1]:
import pandas as pd, numpy as np
from datetime import date, timedelta

# ---------- 1. PRODUCT SETUP ----------

PRODUCTS = [
    {'ASIN': 'B0B94152F6', 'Model_Name': 'MacBook Air M2', 'Release_Year': 2022, 'RAM_GB': 8, 'Storage_GB': 256, 'Color': 'Midnight', 'Base_Price': 110000},
    {'ASIN': 'B0B94213G7', 'Model_Name': 'MacBook Air M2', 'Release_Year': 2022, 'RAM_GB': 16, 'Storage_GB': 512, 'Color': 'Starlight', 'Base_Price': 140000},
    {'ASIN': 'B0CJ5KWD22', 'Model_Name': 'MacBook Pro M3', 'Release_Year': 2023, 'RAM_GB': 8, 'Storage_GB': 512, 'Color': 'Space Grey', 'Base_Price': 169900},
    {'ASIN': 'B0CJ5LSM38', 'Model_Name': 'MacBook Pro M3 Pro', 'Release_Year': 2023, 'RAM_GB': 18, 'Storage_GB': 512, 'Color': 'Space Black', 'Base_Price': 199900},
    {'ASIN': 'B08N5XSG8Z', 'Model_Name': 'MacBook Air M1', 'Release_Year': 2020, 'RAM_GB': 8, 'Storage_GB': 256, 'Color': 'Silver', 'Base_Price': 99900},
]

START_DATE, END_DATE = date(2022, 1, 1), date(2024, 12, 31)
TOTAL_DAYS = (END_DATE - START_DATE).days + 1

APPLE_EVENTS = [date(2022, 9, 7), date(2023, 9, 12), date(2024, 9, 10)]
MAJOR_SALES  = {"Diwali_Sale_Start":(10,20),"Prime_Day_Start":(7,15)}

np.random.seed(42)
all_rows = []

# ---------- 2. MAIN LOOP ----------
for prod in PRODUCTS:
    cost_price = round(prod["Base_Price"] * np.random.uniform(0.70,0.85), -2)
    sales_rank = np.random.randint(5,40)
    total_reviews = np.random.randint(50,500)
    avg_rating = np.random.uniform(4.6,4.9)
    num_sellers = np.random.randint(5,20)
    base_price = prod["Base_Price"]

    for i in range(TOTAL_DAYS):
        d = START_DATE + timedelta(days=i)
        amazon_price = base_price * np.random.uniform(0.9,1.1)
        stock_status = np.random.choice(["In Stock","Low Stock","Temporarily Unavailable"], p=[0.9,0.08,0.02])
        promo_flag = np.random.choice([0,1], p=[0.9,0.1])
        is_fba = np.random.choice([0,1])

        days_until_apple = min([(e-d).days for e in APPLE_EVENTS if e>d]+[999])
        days_until_sale = min([(date(d.year,m,dy)-d).days for (m,dy) in MAJOR_SALES.values() if date(d.year,m,dy)>d]+[999])

        base_demand = 400 if "Air" in prod["Model_Name"] else 550

        # ----- Compute revenue-optimal price -----
        possible_prices = [amazon_price * p for p in [0.9, 0.95, 1.0, 1.05, 1.1]]
        revenues = []

        for p in possible_prices:
            price_factor = (amazon_price / p)
            rating_factor = avg_rating / 5
            sale_boost = 1.3 if days_until_sale < 10 else 1.0
            stock_factor = 1.0 if stock_status=="In Stock" else (0.7 if stock_status=="Low Stock" else 0.2)
            promo_factor = 1.2 if promo_flag==1 else 1.0

            units_sold_sim = base_demand * price_factor * rating_factor * sale_boost * stock_factor * promo_factor * np.random.uniform(0.8,1.2)
            revenue = p * units_sold_sim
            revenues.append(revenue)

        optimal_price = possible_prices[np.argmax(revenues)]

        # ----- Simulate our chosen (current) price scenario -----
        our_price = amazon_price * np.random.choice([0.9,0.95,1.0,1.05,1.1])
        price_diff = our_price - amazon_price
        price_ratio = our_price / amazon_price
        price_lead = "Undercutting" if our_price < amazon_price else ("Matching" if abs(price_diff) < 50 else "Premium")
        is_leader = 1 if our_price < amazon_price else 0

        day_of_week = d.weekday()
        is_weekend  = 1 if day_of_week>=5 else 0
        week_of_year= d.isocalendar().week
        is_month_end= 1 if d.day>26 else 0

        total_reviews += np.random.randint(0,5)
        avg_rating = max(3.5, avg_rating - np.random.uniform(0,0.0005))
        review_velocity_7d = np.random.randint(0,30)
        rating_trend_30d   = np.random.uniform(-0.01,0.01)

        amazon_price_7ma  = amazon_price * np.random.uniform(0.98,1.02)
        amazon_price_30ma = amazon_price * np.random.uniform(0.97,1.03)
        amazon_price_vol30= amazon_price * np.random.uniform(0.005,0.02)

        price_factor = (amazon_price / our_price)
        rating_factor = avg_rating / 5
        sale_boost = 1.3 if days_until_sale<10 else 1.0
        stock_factor = 1.0 if stock_status=="In Stock" else (0.7 if stock_status=="Low Stock" else 0.2)
        promo_factor = 1.2 if promo_flag==1 else 1.0

        units_sold = base_demand * price_factor * rating_factor * sale_boost * stock_factor * promo_factor * np.random.uniform(0.8,1.2)
        units_sold = round(units_sold,2)

        # ---------- Append row ----------
        all_rows.append({
            "ASIN":prod["ASIN"],"Model_Name":prod["Model_Name"],"Release_Year":prod["Release_Year"],
            "RAM_GB":prod["RAM_GB"],"Storage_GB":prod["Storage_GB"],"Color":prod["Color"],
            "Cost_Price":cost_price,
            "Date":d,
            "Amazon_Price":round(amazon_price,2),
            "Stock_Status":stock_status,
            "Sales_Rank":sales_rank + np.random.randint(-3,3),
            "Number_of_Sellers":num_sellers + np.random.randint(-2,2),
            "Promotional_Flag":promo_flag,
            "Is_Amazon_Fulfilled":is_fba,
            "Our_Price":round(our_price,2),
            "Price_Difference":round(price_diff,2),
            "Price_Ratio":round(price_ratio,3),
            "Price_Leadership":price_lead,
            "Is_Price_Leader":is_leader,
            "Amazon_Price_7_Day_Moving_Avg":round(amazon_price_7ma,2),
            "Amazon_Price_30_Day_Moving_Avg":round(amazon_price_30ma,2),
            "Amazon_Price_Volatility_30_Day":round(amazon_price_vol30,2),
            "Day_of_Week":day_of_week,
            "Is_Weekend":is_weekend,
            "Week_of_Year":int(week_of_year),
            "Is_Month_End":is_month_end,
            "Days_Until_Apple_Event":days_until_apple,
            "Days_Until_Major_Sale":days_until_sale,
            "Customer_Rating_Avg":round(avg_rating,2),
            "Total_Reviews_Count":total_reviews,
            "Review_Velocity_7_Day":review_velocity_7d,
            "Rating_Trend_30_Day":round(rating_trend_30d,3),
            "Units_Sold":units_sold,
            "Optimal_Price_Point":round(optimal_price,2)
        })

# ---------- 3. SAVE ----------
df = pd.DataFrame(all_rows)
df.sort_values(["ASIN","Date"], inplace=True)
df.reset_index(drop=True, inplace=True)

out = "synthetic_macbook_regression_revenue_optimal.csv"
df.to_csv(out,index=False)

print(f"✅ Done! Rows: {len(df):,} | Columns: {len(df.columns)}")
print(f"📁 File saved: {out}")
print("\nPreview:")
print(df.head(5))


✅ Done! Rows: 5,480 | Columns: 34
📁 File saved: synthetic_macbook_regression_revenue_optimal.csv

Preview:
         ASIN      Model_Name  Release_Year  RAM_GB  Storage_GB   Color  \
0  B08N5XSG8Z  MacBook Air M1          2020       8         256  Silver   
1  B08N5XSG8Z  MacBook Air M1          2020       8         256  Silver   
2  B08N5XSG8Z  MacBook Air M1          2020       8         256  Silver   
3  B08N5XSG8Z  MacBook Air M1          2020       8         256  Silver   
4  B08N5XSG8Z  MacBook Air M1          2020       8         256  Silver   

   Cost_Price        Date  Amazon_Price Stock_Status  ...  Week_of_Year  \
0     77600.0  2022-01-01      90266.18     In Stock  ...            52   
1     77600.0  2022-01-02     107147.10     In Stock  ...            52   
2     77600.0  2022-01-03     104516.63     In Stock  ...             1   
3     77600.0  2022-01-04     104489.98     In Stock  ...             1   
4     77600.0  2022-01-05      97612.31     In Stock  ...          

In [2]:
# 📦 Import libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ⚙️ Models
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from xgboost import XGBRegressor
from sklearn.svm import SVR

# 📂 Load dataset
data = pd.read_csv("synthetic_macbook_regression_revenue_optimal.csv")

# 🎯 Target and features
X = data.drop(columns=['Optimal_Price_Point'])
y = data['Optimal_Price_Point']

# ⚙️ Identify column types
categorical_cols = X.select_dtypes(include=['object']).columns
numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns

# 🧩 Preprocessor
preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
    ('num', StandardScaler(), numeric_cols)
])

# 🧪 Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 🧱 Models to test
models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(),
    "Lasso Regression": Lasso(),
    "Random Forest": RandomForestRegressor(random_state=42),
    "Extra Trees": ExtraTreesRegressor(random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
    "XGBoost": XGBRegressor(random_state=42, verbosity=0),
    "Support Vector Regressor": SVR()
}

# 📊 Train, evaluate, and collect results
results = []
for name, model in models.items():
    pipe = Pipeline([
        ('preprocess', preprocessor),
        ('model', model)
    ])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)

    mae = mean_absolute_error(y_test, preds)
    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, preds)

    results.append([name, round(r2, 4), round(mae, 2), round(mse, 2), round(rmse, 2)])

# 🧾 Create DataFrame
results_df = pd.DataFrame(results, columns=['Model', 'R2 Score', 'MAE', 'MSE', 'RMSE'])
results_df = results_df.sort_values(by='R2 Score', ascending=False).reset_index(drop=True)

# ✅ Display results in a clean format
print("✅ Model Comparison Results:\n")
print(results_df.to_string(index=False))

# 🏆 Print best model
best_model = results_df.iloc[0]
print("\n🏆 Best Model:")
print(f"Model Name: {best_model['Model']}")
print(f"R² Score: {best_model['R2 Score']}")
print(f"MAE: {best_model['MAE']}")


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_coordinate_descent.py:656: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 93550842496.59537, tolerance: 696266377.2741214
  model = cd_fast.sparse_enet_coordinate_descent(


✅ Model Comparison Results:

                   Model  R2 Score      MAE          MSE     RMSE
       Gradient Boosting    0.9277  8949.41 1.140819e+08 10680.91
           Random Forest    0.9234  9152.84 1.208393e+08 10992.69
             Extra Trees    0.9186  9389.27 1.284920e+08 11335.43
                 XGBoost    0.9183  9418.79 1.288898e+08 11352.96
        Ridge Regression    0.9136  9741.72 1.363896e+08 11678.60
        Lasso Regression    0.9091  9925.41 1.435009e+08 11979.19
       Linear Regression    0.8999 10373.26 1.579466e+08 12567.68
Support Vector Regressor    0.0091 34051.76 1.563534e+09 39541.55

🏆 Best Model:
Model Name: Gradient Boosting
R² Score: 0.9277
MAE: 8949.41
